# Module 01 — Dataset Discovery & Overview

This notebook summarizes the systematic search for publicly available human IVD
single-cell RNA-seq datasets and the resulting inclusion/exclusion decisions.

We searched GEO, ArrayExpress, CellxGene, the Human Cell Atlas Data Portal,
Single Cell Portal, PubMed, and preprint servers using combinations of IVD-related
terms ("intervertebral disc", "nucleus pulposus", "annulus fibrosus", "endplate")
crossed with single-cell technology terms. Results were deduplicated by accession
number and filtered against explicit inclusion/exclusion criteria
(see `specs/01_DATASET_DISCOVERY.md`).

**Manuscript mapping:** Table 1 (dataset characteristics). Methods section on data sources.

**Data source:** `metadata/dataset_registry.tsv` — master list of all candidate datasets.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

BASE = Path('..').resolve()
FIG_DIR = BASE / 'results' / 'qc_reports'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load dataset registry ─────────────────────────────────────────────────
reg = pd.read_csv(BASE / 'metadata' / 'dataset_registry.tsv', sep='\t')
print(f'Total candidate datasets: {len(reg)}')
print(f'  Included: {(reg.status == "included").sum()}')
print(f'  Excluded: {(reg.status == "excluded").sum()}')
print(f'  Deferred: {(reg.status == "deferred").sum()}')

## Full Candidate List

The table below shows every dataset identified during the systematic search.
The **status** column indicates whether each dataset was included in the atlas,
excluded (with reason), or deferred to a later module. Key columns:

- **accession** — GEO/CNGB/other repository identifier
- **compartment** — IVD tissue compartment(s) profiled (NP, AF, CEP, or mixed)
- **technology** — single-cell platform used
- **n_cells_reported** — cell count as reported in the original publication
- **exclusion_reason** — why excluded datasets were dropped

In [ ]:
display_cols = [
    'accession', 'first_author', 'year', 'species', 'compartment',
    'technology', 'n_samples', 'n_cells_reported', 'conditions',
    'status', 'exclusion_reason',
]
styled = (
    reg[display_cols]
    .style
    .apply(
        lambda row: [
            'background-color: #d4edda' if row['status'] == 'included'
            else 'background-color: #f8d7da' if row['status'] == 'excluded'
            else 'background-color: #fff3cd'
        ] * len(row),
        axis=1,
    )
    .set_caption('Dataset Registry — All Candidates')
    .hide(axis='index')
)
display(styled)

## Summary Statistics

The panels below break down the **included** datasets by publication year,
IVD compartment, disease condition, and sequencing platform. These views help
assess whether the atlas has adequate coverage across the axes that matter
for downstream comparisons.

**What to look for:**
- Are any compartments severely underrepresented (especially CEP/endplate)?
- Is there a reasonable balance of healthy vs. degenerated samples?
- Are multiple sequencing platforms represented (relevant for batch effects)?

In [ ]:
inc = reg[reg.status == 'included'].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# A. By publication year
ax = axes[0, 0]
year_counts = inc['year'].value_counts().sort_index()
year_counts.plot.bar(ax=ax, color='#4c72b0', edgecolor='white')
ax.set_title('A. Datasets by publication year')
ax.set_ylabel('Number of datasets')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)

# B. By compartment
ax = axes[0, 1]
# Compartment can have multiple values per study — split on comma/space
compartments = inc['compartment'].str.split(r',\s*|\s+').explode().str.strip()
# Standardize naming
compartment_map = {
    'NP': 'NP', 'AF': 'AF', 'CEP': 'CEP', 'CEP/Endplate': 'CEP',
    'Whole': 'IVD_mixed', 'IVD': 'IVD_mixed',
}
compartments = compartments.map(lambda x: compartment_map.get(x, x))
comp_order = ['NP', 'AF', 'CEP', 'IVD_mixed']
comp_counts = compartments.value_counts().reindex(comp_order, fill_value=0)
comp_counts.plot.bar(ax=ax, color='#55a868', edgecolor='white')
ax.set_title('B. Datasets by IVD compartment')
ax.set_ylabel('Number of datasets')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)

# C. By condition (simplified from the conditions field)
ax = axes[1, 0]
def classify_condition(cond_str):
    cond = str(cond_str).lower()
    tags = []
    if any(w in cond for w in ['healthy', 'normal', 'non-degen']):
        tags.append('Healthy/Normal')
    if any(w in cond for w in ['degen', 'idd', 'herniat', 'modic']):
        tags.append('Degenerated')
    if 'neonatal' in cond:
        tags.append('Neonatal')
    return tags if tags else ['Other']

cond_series = inc['conditions'].apply(classify_condition).explode()
cond_order = ['Healthy/Normal', 'Degenerated', 'Neonatal']
cond_counts = cond_series.value_counts().reindex(cond_order, fill_value=0)
cond_counts.plot.bar(ax=ax, color='#c44e52', edgecolor='white')
ax.set_title('C. Datasets by condition category')
ax.set_ylabel('Number of datasets')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)

# D. By platform
ax = axes[1, 1]
def classify_platform(tech):
    tech = str(tech)
    if '10x' in tech or '10X' in tech or 'Chromium' in tech:
        return '10x Genomics'
    elif 'BD' in tech or 'Rhapsody' in tech:
        return 'BD Rhapsody'
    elif 'Singleron' in tech:
        return 'Singleron'
    else:
        return 'Other'

inc['platform_simple'] = inc['technology'].apply(classify_platform)
plat_counts = inc['platform_simple'].value_counts()
plat_counts.plot.bar(ax=ax, color='#8172b2', edgecolor='white')
ax.set_title('D. Datasets by sequencing platform')
ax.set_ylabel('Number of datasets')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)

fig.suptitle('Included Dataset Characteristics', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / 'notebook_01_dataset_summary.png')
plt.show()

## Cell Counts per Study

The bar chart below shows the reported cell count for each included dataset.
This gives a sense of each study's contribution to the final atlas. Larger
datasets will have more statistical weight in integrated analyses, and
very small datasets (< 1,000 cells) may contribute noise without adding
much power.

Note: these are *reported* cell counts from the original publications,
not post-QC counts (see `notebooks/03_qc.ipynb` for the latter).

In [ ]:
# Parse cell counts — some are NA or non-numeric
inc_cells = inc[['accession', 'first_author', 'year', 'n_cells_reported']].copy()
inc_cells['n_cells_reported'] = pd.to_numeric(inc_cells['n_cells_reported'], errors='coerce')
inc_cells = inc_cells.sort_values('n_cells_reported', ascending=True)
inc_cells['label'] = inc_cells['accession'] + ' (' + inc_cells['first_author'] + ')'

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    inc_cells['label'], inc_cells['n_cells_reported'],
    color='#4c72b0', edgecolor='white',
)
# Annotate bars
for bar, val in zip(bars, inc_cells['n_cells_reported']):
    if not np.isnan(val):
        ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height() / 2,
                f'{int(val):,}', va='center', fontsize=8)
    else:
        ax.text(500, bar.get_y() + bar.get_height() / 2,
                'not reported', va='center', fontsize=8, color='gray')

ax.set_xlabel('Reported cell count')
ax.set_title('Reported Cell Counts per Included Dataset')
fig.tight_layout()
fig.savefig(FIG_DIR / 'notebook_01_cell_counts.png')
plt.show()

total_cells = inc_cells['n_cells_reported'].sum()
print(f'Total reported cells across all datasets: {int(total_cells):,}')
print(f'Datasets with missing cell counts: {inc_cells["n_cells_reported"].isna().sum()}')

## Condition x Compartment Coverage

A key question for any multi-study atlas is whether the available data covers the
biological space of interest. The heatmap below maps each included dataset to its
IVD compartment(s) and disease condition(s), revealing:

- **Well-covered regions** (e.g., degenerated NP) — where multiple studies contribute data
- **Gaps** (e.g., healthy endplate, neonatal AF) — where no datasets exist
- **Sparse regions** — covered by only one study, limiting cross-study validation

These gaps are important context for interpreting downstream results: absence of
evidence in a gap region does not mean absence of biology, just absence of data.

In [ ]:
# Load sample metadata for finer-grained condition x compartment mapping
meta = pd.read_csv(BASE / 'metadata' / 'sample_metadata.tsv', sep='\t')

# Build coverage matrix: condition_harmonized x compartment
cond_order = ['healthy', 'degenerated_mild', 'degenerated_severe',
              'degenerated_ungraded', 'herniated', 'neonatal', 'aged_ungraded']
comp_order = ['NP', 'AF', 'CEP', 'IVD_mixed']

# Count number of datasets (not samples) per condition x compartment
coverage = (
    meta.groupby(['condition_harmonized', 'compartment'])['study_accession']
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=cond_order, columns=comp_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    coverage, annot=True, fmt='d', cmap='YlOrRd', linewidths=1, linecolor='white',
    cbar_kws={'label': 'Number of datasets'}, ax=ax,
)
ax.set_xlabel('IVD Compartment')
ax.set_ylabel('Condition (harmonized)')
ax.set_title('Dataset Coverage: Condition x Compartment')
ax.tick_params(axis='y', rotation=0)

fig.tight_layout()
fig.savefig(FIG_DIR / 'notebook_01_coverage_heatmap.png')
plt.show()

# Identify gaps
gaps = []
for cond in cond_order:
    for comp in comp_order:
        if coverage.loc[cond, comp] == 0:
            gaps.append(f'{cond} x {comp}')
if gaps:
    print(f'Coverage gaps ({len(gaps)}): {", ".join(gaps)}')
else:
    print('No coverage gaps — all condition x compartment combinations have data.')

## Exclusion Reasons

Understanding *why* datasets were excluded is important for assessing potential
bias in the atlas. The table below groups excluded datasets by reason.

In [ ]:
exc = reg[reg.status == 'excluded'].copy()
exc_display = exc[['accession', 'first_author', 'year', 'species', 'exclusion_reason']].copy()
exc_display = exc_display.fillna({'accession': 'N/A'})
display(
    exc_display
    .style
    .set_caption('Excluded Datasets and Reasons')
    .hide(axis='index')
)

print(f'\nExcluded: {len(exc)} datasets')
print('Breakdown:')
for reason, group in exc.groupby('exclusion_reason'):
    print(f'  {reason}: {len(group)}')

## Notes

- The full search strategy and query log are documented in `metadata/search_log.md`.
- One dataset (Zhou 2023, embryonic IVD) was **deferred** rather than excluded — it may
  be revisited in Module 08 (trajectory analysis) for notochordal-to-NP differentiation.
- Raw data for all included datasets is stored in `data/raw/{accession}/`.
- Figures saved to `results/qc_reports/` with prefix `notebook_01_`.

In [ ]:
saved_figures = sorted(FIG_DIR.glob('notebook_01_*.png'))
print('Saved figures:')
for f in saved_figures:
    print(f'  {f.relative_to(BASE)}')